In [1]:
"""
=============================================================
DSA COMPFEST 18 — Sigma3 V3 (Knowledge-Enhanced)
=============================================================
Teknik baru dari literatur:
1. Box-Cox transform target → normalitas p=0.997 vs log1p p≈0
2. Pseudo-labeling → kurangi distribusi gap train-test
3. LGB + XGB + CatBoost 3-way blend
4. Optuna 30 trials per model
5. Fitur: sigma3 EXACT (6 flags, zero tambahan)
=============================================================
"""

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import boxcox
from scipy.special import inv_boxcox
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================================================
# KONFIGURASI
# ============================================================
SEED       = 42
N_FOLDS    = 5
TUNE_FOLDS = 3
N_TRIALS   = 30
TARGET     = 'property_organic_content'
ID_COL     = 'sample_id'
np.random.seed(SEED)

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

# ============================================================
# LOAD DATA
# ============================================================
DATA_DIR = '/kaggle/input/competitions/seleksi-data-science-academy-compfest-18'
OUT_DIR  = '/kaggle/working/'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
train_ids = train[ID_COL].copy()
test_ids  = test[ID_COL].copy()
print(f"Train: {train.shape} | Test: {test.shape}")

# ============================================================
# TARGET TRANSFORM — Box-Cox
# Box-Cox: skew=0.0004, normaltest p=0.997
# log1p : skew=0.142,  normaltest p≈0
# Box-Cox jauh lebih mendekati normal → optimal untuk RMSE
# ============================================================
y_raw = train[TARGET].values
y_bc, LAMBDA = boxcox(y_raw + 1)
print(f"Box-Cox λ={LAMBDA:.4f} | skew={pd.Series(y_bc).skew():.6f}")

def inverse_transform(x):
    return np.clip(inv_boxcox(x, LAMBDA) - 1, 0, None)

y = pd.Series(y_bc)

# ============================================================
# FEATURE ENGINEERING — sigma3 EXACT (6 flags saja)
# Terbukti dari semua submission: setiap tambahan fitur → LB naik
# ============================================================
CAT_COLS = [
    'source_id', 'has_band_A_spectrum', 'has_band_B_spectrum',
    'sampling_strategy', 'sampling_depth_cm', 'geo_zone_macro',
    'geo_zone_micro', 'geo_zone_meso', 'land_cover_type',
    'biome', 'parent_rock_type'
]

def feature_engineer(df):
    df = df.copy()
    df['flag_has_coord']     = df['latitude'].notna().astype(int)
    df['flag_has_acidity']   = df['property_acidity_index'].notna().astype(int)
    df['flag_has_cation_na'] = df['cation_Na'].notna().astype(int)
    df['flag_has_cation_set']= df['cation_Ca'].notna().astype(int)
    df['flag_has_particle']  = df['property_particle_coarse'].notna().astype(int)
    df['flag_has_band_B']    = (df['has_band_B_spectrum'] == 'YES').astype(int)
    for c in CAT_COLS:
        df[c] = df[c].astype('category')
    return df

train_fe = feature_engineer(train)
test_fe  = feature_engineer(test)

# Align categories train & test
for c in CAT_COLS:
    cats = pd.api.types.union_categoricals(
        [train_fe[c], test_fe[c]]).categories
    train_fe[c] = train_fe[c].cat.set_categories(cats)
    test_fe[c]  = test_fe[c].cat.set_categories(cats)

FEATURE_COLS = [c for c in train_fe.columns if c not in [ID_COL, TARGET]]
X      = train_fe[FEATURE_COLS]
X_test = test_fe[FEATURE_COLS]
print(f"Total fitur: {X.shape[1]}")

# XGBoost: integer codes
tx = train_fe.copy(); ttx = test_fe.copy()
for c in CAT_COLS:
    tx[c]  = tx[c].cat.codes.replace(-1, np.nan)
    ttx[c] = ttx[c].cat.codes.replace(-1, np.nan)
X_xgb = tx[FEATURE_COLS]; X_test_xgb = ttx[FEATURE_COLS]

# CatBoost: string
tbc = train_fe.copy(); ttbc = test_fe.copy()
for c in CAT_COLS:
    tbc[c]  = tbc[c].astype(str).fillna('Unknown')
    ttbc[c] = ttbc[c].astype(str).fillna('Unknown')
X_cb = tbc[FEATURE_COLS]; X_test_cb = ttbc[FEATURE_COLS]

# ============================================================
# CV SETUP
# ============================================================
kf      = KFold(n_splits=N_FOLDS,    shuffle=True, random_state=SEED)
tune_kf = KFold(n_splits=TUNE_FOLDS, shuffle=True, random_state=SEED)

# ============================================================
# OPTUNA — LightGBM
# Nama parameter LENGKAP langsung, tanpa string replace
# ============================================================
def lgb_objective(trial):
    params = {
        'objective'        : 'regression',
        'metric'           : 'rmse',
        'verbosity'        : -1,
        'seed'             : SEED,
        'bagging_freq'     : 1,
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 15, 200),
        'min_data_in_leaf' : trial.suggest_int('min_data_in_leaf', 5, 80),
        'feature_fraction' : trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction' : trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'lambda_l1'        : trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2'        : trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
    }
    oof = np.zeros(len(X))
    for ti, vi in tune_kf.split(X):
        dtr  = lgb.Dataset(X.iloc[ti], label=y.iloc[ti],
                           categorical_feature=CAT_COLS)
        dval = lgb.Dataset(X.iloc[vi], label=y.iloc[vi],
                           categorical_feature=CAT_COLS, reference=dtr)
        m = lgb.train(params, dtr, num_boost_round=2000, valid_sets=[dval],
                      callbacks=[lgb.early_stopping(100, verbose=False),
                                 lgb.log_evaluation(0)])
        oof[vi] = m.predict(X.iloc[vi], num_iteration=m.best_iteration)
    return rmse(y, oof)

print("\nTuning LGB...")
lgb_study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS)
print(f"  Best: {lgb_study.best_value:.5f} | params: {lgb_study.best_params}")

# ============================================================
# OPTUNA — XGBoost
# ============================================================
def xgb_objective(trial):
    params = {
        'objective'            : 'reg:squarederror',
        'tree_method'          : 'hist',
        'enable_categorical'   : False,
        'random_state'         : SEED,
        'n_estimators'         : 1500,
        'early_stopping_rounds': 50,
        'n_jobs'               : -1,
        'verbosity'            : 0,
        'learning_rate'        : trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'max_depth'            : trial.suggest_int('max_depth', 3, 10),
        'min_child_weight'     : trial.suggest_int('min_child_weight', 1, 20),
        'subsample'            : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'     : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'            : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda'           : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    oof = np.zeros(len(X_xgb))
    for ti, vi in tune_kf.split(X_xgb):
        m = xgb.XGBRegressor(**params)
        m.fit(X_xgb.iloc[ti], y.iloc[ti],
              eval_set=[(X_xgb.iloc[vi], y.iloc[vi])], verbose=False)
        oof[vi] = m.predict(X_xgb.iloc[vi])
    return rmse(y, oof)

print("\nTuning XGB...")
xgb_study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS)
print(f"  Best: {xgb_study.best_value:.5f} | params: {xgb_study.best_params}")

# ============================================================
# OPTUNA — CatBoost
# Nama parameter LENGKAP — tidak ada ambiguitas
# ============================================================
def cb_objective(trial):
    params = {
        'iterations'         : 1500,
        'loss_function'      : 'RMSE',
        'eval_metric'        : 'RMSE',
        'random_seed'        : SEED,
        'verbose'            : False,
        'allow_writing_files': False,
        'learning_rate'      : trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'depth'              : trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg'        : trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
        'subsample'          : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bylevel'  : trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'min_data_in_leaf'   : trial.suggest_int('min_data_in_leaf', 1, 50),
    }
    oof = np.zeros(len(X_cb))
    for ti, vi in tune_kf.split(X_cb):
        m = cb.CatBoostRegressor(**params)
        m.fit(X_cb.iloc[ti], y.iloc[ti],
              cat_features=CAT_COLS,
              eval_set=(X_cb.iloc[vi], y.iloc[vi]),
              early_stopping_rounds=80, verbose=False)
        oof[vi] = m.predict(X_cb.iloc[vi])
    return rmse(y, oof)

print("\nTuning CB...")
cb_study = optuna.create_study(direction='minimize',
                               sampler=optuna.samplers.TPESampler(seed=SEED))
cb_study.optimize(cb_objective, n_trials=N_TRIALS)
print(f"  Best: {cb_study.best_value:.5f} | params: {cb_study.best_params}")

# ============================================================
# FINAL PARAMS — langsung dari study.best_params, tanpa remapping
# ============================================================
lgb_p = {
    'objective': 'regression', 'metric': 'rmse',
    'verbosity': -1, 'seed': SEED, 'bagging_freq': 1,
    **lgb_study.best_params
}
xgb_p = {
    'objective': 'reg:squarederror', 'tree_method': 'hist',
    'enable_categorical': False, 'random_state': SEED,
    'n_estimators': 3000, 'early_stopping_rounds': 100,
    'n_jobs': -1, 'verbosity': 0,
    **xgb_study.best_params
}
cb_p = {
    'iterations': 3000, 'loss_function': 'RMSE', 'eval_metric': 'RMSE',
    'random_seed': SEED, 'verbose': False, 'allow_writing_files': False,
    **cb_study.best_params
}

# ============================================================
# PHASE 1 — Training awal untuk generate pseudo-labels
# ============================================================
print("\n--- PHASE 1: Training awal ---")
oof_l1 = np.zeros(len(X));     tst_l1 = np.zeros(len(X_test))
oof_x1 = np.zeros(len(X_xgb)); tst_x1 = np.zeros(len(X_test_xgb))
oof_c1 = np.zeros(len(X_cb));  tst_c1 = np.zeros(len(X_test_cb))

for fi, (ti, vi) in enumerate(kf.split(X)):
    # LGB
    dtr  = lgb.Dataset(X.iloc[ti], label=y.iloc[ti],
                       categorical_feature=CAT_COLS)
    dval = lgb.Dataset(X.iloc[vi], label=y.iloc[vi],
                       categorical_feature=CAT_COLS, reference=dtr)
    m = lgb.train(lgb_p, dtr, num_boost_round=3000, valid_sets=[dval],
                  callbacks=[lgb.early_stopping(100, verbose=False),
                             lgb.log_evaluation(0)])
    oof_l1[vi] = m.predict(X.iloc[vi], num_iteration=m.best_iteration)
    tst_l1    += m.predict(X_test, num_iteration=m.best_iteration) / N_FOLDS

    # XGB
    mx = xgb.XGBRegressor(**xgb_p)
    mx.fit(X_xgb.iloc[ti], y.iloc[ti],
           eval_set=[(X_xgb.iloc[vi], y.iloc[vi])], verbose=False)
    oof_x1[vi] = mx.predict(X_xgb.iloc[vi])
    tst_x1    += mx.predict(X_test_xgb) / N_FOLDS

    # CB
    mc = cb.CatBoostRegressor(**cb_p)
    mc.fit(X_cb.iloc[ti], y.iloc[ti],
           cat_features=CAT_COLS,
           eval_set=(X_cb.iloc[vi], y.iloc[vi]),
           early_stopping_rounds=100, verbose=False)
    oof_c1[vi] = mc.predict(X_cb.iloc[vi])
    tst_c1    += mc.predict(X_test_cb) / N_FOLDS

    fl = rmse(y.iloc[vi], oof_l1[vi])
    fx = rmse(y.iloc[vi], oof_x1[vi])
    fc = rmse(y.iloc[vi], oof_c1[vi])
    print(f"  Fold {fi+1}: LGB={fl:.5f} | XGB={fx:.5f} | CB={fc:.5f}")

print(f"\nPhase 1 OOF: LGB={rmse(y,oof_l1):.5f} XGB={rmse(y,oof_x1):.5f} CB={rmse(y,oof_c1):.5f}")

# Grid search blend untuk pseudo-labels
bw, br = (0.75, 0.25, 0.0), 1e9
for wl in np.arange(0, 1.01, 0.05):
    for wx in np.arange(0, 1.01-wl, 0.05):
        wc = round(1.0 - wl - wx, 10)
        if wc < 0: continue
        r = rmse(y, wl*oof_l1 + wx*oof_x1 + wc*oof_c1)
        if r < br: br, bw = r, (wl, wx, wc)

wl, wx, wc = bw
pseudo_test_bc = wl*tst_l1 + wx*tst_x1 + wc*tst_c1
print(f"Phase 1 blend: LGB={wl:.2f} XGB={wx:.2f} CB={wc:.2f} → OOF={br:.5f}")

# ============================================================
# PHASE 2 — Pseudo-labeling
# Tambahkan test dengan pseudo-labels ke training data
# Model "melihat" distribusi test → kurangi covariate shift residual
# ============================================================
print("\n--- PHASE 2: Pseudo-labeling ---")

# Augmented datasets
X_aug     = pd.concat([X,     test_fe[FEATURE_COLS]], ignore_index=True)
X_xgb_aug = pd.concat([X_xgb, ttx[FEATURE_COLS]],    ignore_index=True)
X_cb_aug  = pd.concat([X_cb,  ttbc[FEATURE_COLS]],    ignore_index=True)
y_aug     = pd.Series(np.concatenate([y.values, pseudo_test_bc]))

print(f"Augmented size: {len(X_aug)} (train={len(X)}, pseudo={len(X_test)})")

oof_l2 = np.zeros(len(X));     tst_l2 = np.zeros(len(X_test))
oof_x2 = np.zeros(len(X_xgb)); tst_x2 = np.zeros(len(X_test_xgb))
oof_c2 = np.zeros(len(X_cb));  tst_c2 = np.zeros(len(X_test_cb))

for fi, (ti, vi) in enumerate(kf.split(X_aug)):
    # Validasi HANYA pada original train (index < len(X))
    orig_vi = vi[vi < len(X)]
    if len(orig_vi) == 0:
        continue

    # LGB — train pada augmented, validasi pada original
    dtr  = lgb.Dataset(X_aug.iloc[ti], label=y_aug.iloc[ti],
                       categorical_feature=CAT_COLS)
    dval = lgb.Dataset(X.iloc[orig_vi], label=y.iloc[orig_vi],
                       categorical_feature=CAT_COLS, reference=dtr)
    m = lgb.train(lgb_p, dtr, num_boost_round=3000, valid_sets=[dval],
                  callbacks=[lgb.early_stopping(100, verbose=False),
                             lgb.log_evaluation(0)])
    oof_l2[orig_vi] = m.predict(X.iloc[orig_vi], num_iteration=m.best_iteration)
    tst_l2         += m.predict(X_test, num_iteration=m.best_iteration) / N_FOLDS

    # XGB
    mx = xgb.XGBRegressor(**xgb_p)
    mx.fit(X_xgb_aug.iloc[ti], y_aug.iloc[ti],
           eval_set=[(X_xgb.iloc[orig_vi], y.iloc[orig_vi])], verbose=False)
    oof_x2[orig_vi] = mx.predict(X_xgb.iloc[orig_vi])
    tst_x2         += mx.predict(X_test_xgb) / N_FOLDS

    # CB
    mc = cb.CatBoostRegressor(**cb_p)
    mc.fit(X_cb_aug.iloc[ti], y_aug.iloc[ti],
           cat_features=CAT_COLS,
           eval_set=(X_cb.iloc[orig_vi], y.iloc[orig_vi]),
           early_stopping_rounds=100, verbose=False)
    oof_c2[orig_vi] = mc.predict(X_cb.iloc[orig_vi])
    tst_c2         += mc.predict(X_test_cb) / N_FOLDS

    fl = rmse(y.iloc[orig_vi], oof_l2[orig_vi])
    fx = rmse(y.iloc[orig_vi], oof_x2[orig_vi])
    fc = rmse(y.iloc[orig_vi], oof_c2[orig_vi])
    print(f"  Fold {fi+1}: LGB={fl:.5f} | XGB={fx:.5f} | CB={fc:.5f}")

print(f"\nPhase 2 OOF: LGB={rmse(y,oof_l2):.5f} XGB={rmse(y,oof_x2):.5f} CB={rmse(y,oof_c2):.5f}")

# ============================================================
# GRID SEARCH 3-WAY BLEND — Phase 2
# ============================================================
best_w, best_r = (0.75, 0.25, 0.0), 1e9
for wl in np.arange(0.0, 1.01, 0.05):
    for wx in np.arange(0.0, 1.01-wl, 0.05):
        wc = round(1.0 - wl - wx, 10)
        if wc < 0: continue
        r = rmse(y, wl*oof_l2 + wx*oof_x2 + wc*oof_c2)
        if r < best_r:
            best_r, best_w = r, (wl, wx, wc)

wl, wx, wc = best_w
oof_final = wl*oof_l2 + wx*oof_x2 + wc*oof_c2
print(f"\nBest blend: LGB={wl:.2f} XGB={wx:.2f} CB={wc:.2f}")
print(f"OOF BC-RMSE  : {best_r:.5f}")
print(f"Phase 1 blend: {br:.5f}  (benchmark)")

oof_orig = inverse_transform(oof_final)
tst_orig = inverse_transform(wl*tst_l2 + wx*tst_x2 + wc*tst_c2)
print(f"OOF RMSE asli: {rmse(y_raw, oof_orig):.4f}")

# ============================================================
# SUBMISSION
# ============================================================
sub = pd.DataFrame({ID_COL: test_ids, TARGET: np.clip(tst_orig, 0, None)})
assert len(sub) == 2670
assert sub[TARGET].isna().sum() == 0

sub.to_csv('submission.csv', index=False)
print(f"\nsubmission.csv saved")
print(sub.describe())

pd.DataFrame({
    'sample_id': train_ids,
    'y_true'   : y_raw,
    'oof_pred' : oof_orig,
}).to_csv('oof_predictions.csv', index=False)
print("oof_predictions.csv saved")

Train: (11210, 52) | Test: (2670, 51)
Box-Cox λ=-0.0818 | skew=0.000396
Total fitur: 56

Tuning LGB...
  Best: 0.22278 | params: {'learning_rate': 0.010105352728014797, 'num_leaves': 169, 'min_data_in_leaf': 70, 'feature_fraction': 0.563268268734059, 'bagging_fraction': 0.9415305668712989, 'lambda_l1': 0.11523251162387899, 'lambda_l2': 8.68620180389913}

Tuning XGB...
  Best: 0.22678 | params: {'learning_rate': 0.018100027978576773, 'max_depth': 9, 'min_child_weight': 18, 'subsample': 0.7020843771180812, 'colsample_bytree': 0.5646131296003669, 'reg_alpha': 0.01477974473059982, 'reg_lambda': 0.0037209487301351368}

Tuning CB...
  Best: 0.22581 | params: {'learning_rate': 0.04065790564059039, 'depth': 8, 'l2_leaf_reg': 1.9086997564048975, 'subsample': 0.5565757572232559, 'colsample_bylevel': 0.8709496262914145, 'min_data_in_leaf': 14}

--- PHASE 1: Training awal ---
  Fold 1: LGB=0.22111 | XGB=0.22449 | CB=0.22174
  Fold 2: LGB=0.21392 | XGB=0.22142 | CB=0.22272
  Fold 3: LGB=0.21623 | X